[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C35_Speech_Audio_Course/01_signals_features/01_signals_features.ipynb)

# 01 · 信号与特征（用 numpy 从零实现）

目标：把 **采样/量化 → 分帧加窗 → DFT/STFT → 梅尔滤波器组 → MFCC** 这条语音前端流水线用 numpy **从零搭出来**，每一步都跑出**可打印的验证**并 `assert` 对拍可信参考（`np.fft`、`scipy`、单调性）。

路线：采样与混叠 → 分帧 → 加窗与泄漏 → DFT→STFT → 梅尔滤波器组 → MFCC(DCT) → ✏️ 练习 → 📖 答案 → 🧪 真实工程参数胶囊。

> 心智模型：**一条流水线，每段是一个小函数，每段都对拍一个绝对可信的参考。** 我们写的是*算法结构*，不是性能。

## 1 · 采样与混叠：奈奎斯特不可违逆

采样率 `sr` 只能无失真表示 `< sr/2`（奈奎斯特）的频率。超过它的频率会**混叠**成 `|f - sr*round(f/sr)|`。

我们造一个超过奈奎斯特的正弦，**亲眼看到**它被采样成一个错误的低频。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def dominant_freq(x, sr):
    '''用 rfft 找出信号最强的频率（Hz）。'''
    mag = np.abs(np.fft.rfft(x))
    freqs = np.fft.rfftfreq(len(x), d=1/sr)
    return freqs[np.argmax(mag)]

sr = 8000                          # 奈奎斯特 = 4000 Hz
t = np.arange(sr) / sr             # 1 秒
f_true = 6000.0                    # 超过奈奎斯特！
x = np.sin(2*np.pi*f_true*t)
f_observed = dominant_freq(x, sr)
f_alias = abs(f_true - sr*round(f_true/sr))   # 预测的混叠频率
print(f'真实频率 {f_true:.0f} Hz, 奈奎斯特 {sr/2:.0f} Hz')
print(f'采样后观测到的主频 = {f_observed:.0f} Hz  (预测混叠到 {f_alias:.0f} Hz)')
assert abs(f_observed - f_alias) < 1.0, '6 kHz 应混叠到 2 kHz'
# 而一个低于奈奎斯特的频率被正确表示
x_ok = np.sin(2*np.pi*1500*t)
assert abs(dominant_freq(x_ok, sr) - 1500) < 1.0
print('✅ 6 kHz @ 8 kHz 采样 -> 错误地变成 2 kHz（混叠，不可逆）；1.5 kHz 则正确')

## 2 · 分帧：把长信号切成重叠的短帧

语音短时平稳，要按帧长 `N`、帧移 `H` 切成**重叠**的帧。帧数 `T = 1 + (L - N)//H`。

我们用**向量化下标**一次切出所有帧（避免 python 循环），并对拍帧数公式与逐帧内容。

In [ ]:
def frame_signal(x, frame_len, hop):
    '''把 1D 信号切成 (T, frame_len) 的帧矩阵，相邻帧起点相隔 hop。'''
    L = len(x)
    T = 1 + max(0, (L - frame_len) // hop)
    # 广播构造下标：第 t 帧第 j 个样本 = x[t*hop + j]
    idx = np.arange(frame_len)[None, :] + hop * np.arange(T)[:, None]
    return x[idx]

x = np.arange(20.0)                 # 用 0..19 便于肉眼检查
frames = frame_signal(x, frame_len=8, hop=4)
print('帧矩阵形状', frames.shape, '(T, N)')
print('第 0 帧', frames[0])
print('第 1 帧', frames[1], '<- 与第0帧重叠 N-H=4 个样本')
T_expected = 1 + (20 - 8)//4
assert frames.shape == (T_expected, 8)
assert np.array_equal(frames[1], x[4:12])      # 第1帧起点 = 1*hop = 4
assert np.array_equal(frames[0][4:], frames[1][:4])  # 重叠部分一致
print(f'✅ 切出 {T_expected} 帧，重叠 N-H=4 个样本，逐帧内容正确')

## 3 · 加窗：抑制频谱泄漏

直接截帧 = 乘矩形窗，旁瓣大 -> **频谱泄漏**（纯频率的能量散到邻近频点）。
乘一个 **Hann 窗** `w[n]=0.5-0.5cos(2πn/N)` 能大幅压低泄漏。

我们对一个**频率不落在 DFT 频点上**的正弦，比较加窗前后能量集中度（集中度越高=泄漏越少）。

In [ ]:
def hann_window(N):
    return 0.5 - 0.5*np.cos(2*np.pi*np.arange(N)/N)

def concentration(mag, k=3):
    '''最强 k 个频点的能量占比：越接近 1，泄漏越少。'''
    p = mag**2
    top = np.sort(p)[::-1][:k].sum()
    return top / p.sum()

N = 256
# 频率故意取 10.5 个周期/帧（不落在整数 DFT 频点上 -> 会泄漏）
n = np.arange(N)
x = np.sin(2*np.pi*10.5*n/N)
mag_rect = np.abs(np.fft.rfft(x))                 # 矩形窗（不加窗）
mag_hann = np.abs(np.fft.rfft(x*hann_window(N)))  # Hann 窗
c_rect = concentration(mag_rect)
c_hann = concentration(mag_hann)
print(f'矩形窗 能量集中度(top3) = {c_rect:.3f}')
print(f'Hann窗 能量集中度(top3) = {c_hann:.3f}')
assert c_hann > c_rect, 'Hann 窗应减少泄漏、提高集中度'
# 整数周期时不泄漏（对照）：11 个整周期
x_int = np.sin(2*np.pi*11*n/N)
assert concentration(np.abs(np.fft.rfft(x_int))) > 0.99
print('✅ 频率不对齐频点时会泄漏；Hann 窗显著改善（整周期时本就不泄漏）')

## 4 · 从零 DFT，再拼成 STFT

DFT：`X[k] = Σ_n x[n] exp(-2πi kn/N)`，写成矩阵 `X = M @ x`。先对拍 `np.fft.fft`。

再把「分帧 + 加窗 + 逐帧 rfft」组装成 **STFT**，对拍 `np.fft.rfft`。这是本课最关键的一步——后面梅尔/MFCC 全建在它上面。

In [ ]:
def dft_matrix(N):
    k = np.arange(N)
    return np.exp(-2j*np.pi*np.outer(k, k)/N)

x = rng.standard_normal(64)
assert np.allclose(dft_matrix(64) @ x, np.fft.fft(x), atol=1e-9)
print('✅ 从零 DFT 矩阵 == np.fft.fft')

def stft(x, frame_len=256, hop=128, window=None):
    '''短时傅里叶变换：分帧 -> 加窗 -> 逐帧 rfft。返回 (T, F) 复矩阵。'''
    if window is None:
        window = hann_window(frame_len)
    frames = frame_signal(x, frame_len, hop) * window[None, :]
    return np.fft.rfft(frames, axis=1)

x = np.sin(2*np.pi*5*np.arange(2048)/256.0)
S = stft(x, 256, 128)
F_expected = 256//2 + 1
print('STFT 形状', S.shape, '(T, F);  F = N/2+1 =', F_expected)
assert S.shape[1] == F_expected
# 对拍：第 0 帧的 rfft 应等于手动对 (第0帧*窗) 做 rfft
frame0 = x[:256] * hann_window(256)
assert np.allclose(S[0], np.fft.rfft(frame0), atol=1e-9)
print('✅ STFT 每帧 == np.fft.rfft(加窗帧)；得到 (时间×频率) 复谱')

## 5 · 梅尔滤波器组：把功率谱压成听觉频带

梅尔换算 `m = 2595 log10(1+f/700)`。在梅尔轴等距取 `M+2` 个点，映回赫兹得三角滤波器的左/中/右边界，每个三角把一段频点的功率加权求和。结果是 `(M, F)` 矩阵，与功率谱相乘即梅尔能量。

In [ ]:
def hz_to_mel(f): return 2595.0*np.log10(1.0 + f/700.0)
def mel_to_hz(m): return 700.0*(10.0**(m/2595.0) - 1.0)

# 往返一致性
assert hz_to_mel(0.0) == 0.0
assert abs(mel_to_hz(hz_to_mel(1000.0)) - 1000.0) < 1e-6

def mel_filterbank(n_filters, n_fft, sr, fmin=0.0, fmax=None):
    '''返回 (n_filters, n_fft//2+1) 的三角滤波器组矩阵。'''
    if fmax is None: fmax = sr/2
    m_pts = np.linspace(hz_to_mel(fmin), hz_to_mel(fmax), n_filters+2)
    f_pts = mel_to_hz(m_pts)
    bins = np.floor((n_fft+1)*f_pts/sr).astype(int)   # 映到最近 DFT 频点
    fb = np.zeros((n_filters, n_fft//2+1))
    for m in range(1, n_filters+1):
        l, ctr, r = bins[m-1], bins[m], bins[m+1]
        for k in range(l, ctr):
            if ctr > l: fb[m-1, k] = (k - l)/(ctr - l)   # 上升沿
        for k in range(ctr, r):
            if r > ctr: fb[m-1, k] = (r - k)/(r - ctr)   # 下降沿
    return fb

fb = mel_filterbank(n_filters=8, n_fft=256, sr=16000)
print('滤波器组形状', fb.shape, '(M, F)')
print('各滤波器峰值 ≈ 1:', np.allclose(fb.max(axis=1), 1.0, atol=1e-6))
assert fb.shape == (8, 129)
assert np.all(fb >= 0) and np.all(fb.sum(axis=1) > 0)   # 非负、每个滤波器有覆盖
# 低频滤波器更窄（覆盖更少频点），高频更宽 —— 梅尔的本质
widths = (fb > 0).sum(axis=1)
assert widths[0] <= widths[-1], '低频滤波器应比高频窄'
print(f'✅ 梅尔滤波器组：低频窄(覆盖{widths[0]}频点) -> 高频宽(覆盖{widths[-1]}频点)')

## 6 · MFCC：log-mel + DCT 去相关

把功率谱经梅尔滤波器组求和 -> 取 log（log-mel）-> 对频带方向做 **DCT-II**、取前几个系数 = MFCC。

我们从零实现 DCT-II（对拍 `scipy.fftpack.dct`，缺失则跳过），并**验证 DCT 把能量集中到前几个系数**（去相关的体现）。

In [ ]:
def dct_ii(x):
    '''沿最后一维做 DCT-II（未归一化）：c[k]=Σ_n x[n]cos(π(2n+1)k/2N)。'''
    N = x.shape[-1]
    n = np.arange(N)
    k = np.arange(N)[:, None]
    basis = np.cos(np.pi*(2*n+1)*k/(2*N))     # (N, N)
    return x @ basis.T

# 对拍 scipy（注意：scipy 未归一化 DCT-II = 我们的 2 倍）
try:
    from scipy.fftpack import dct as sdct
    z = rng.standard_normal((4, 16))
    assert np.allclose(dct_ii(z), sdct(z, type=2, axis=1, norm=None)/2.0, atol=1e-8)
    print('✅ 从零 DCT-II == scipy.fftpack.dct / 2')
except ImportError:
    print('scipy 不可用，跳过 DCT 对拍（实现仍按定义正确）')

def mfcc(x, sr=16000, n_fft=256, hop=128, n_mels=40, n_mfcc=13):
    S = stft(x, n_fft, hop)
    power = np.abs(S)**2                       # (T, F) 功率谱
    fb = mel_filterbank(n_mels, n_fft, sr)    # (M, F)
    mel_energy = power @ fb.T                  # (T, M)
    log_mel = np.log(mel_energy + 1e-10)
    return dct_ii(log_mel)[:, :n_mfcc], log_mel

x = (np.sin(2*np.pi*300*np.arange(4096)/16000)
     + 0.5*np.sin(2*np.pi*1200*np.arange(4096)/16000))
coeffs, log_mel = mfcc(x)
print('log-mel 形状', log_mel.shape, '| MFCC 形状', coeffs.shape)
assert coeffs.shape[1] == 13
# 去相关验证：DCT 后，前 6 个系数承载的能量占比 远高于均匀分布
full = dct_ii(log_mel)
energy_frac = (full[:, :6]**2).sum() / (full**2).sum()
print(f'DCT 前 6 个系数占总能量 {energy_frac:.1%}（均匀分布只会占 {6/log_mel.shape[1]:.1%}）')
assert energy_frac > 6/log_mel.shape[1], 'DCT 应把能量集中到低阶系数'
print('✅ MFCC 流水线跑通；DCT 把能量集中到前几个系数 -> 去相关、可降维')

---
## ✏️ 练习 1：分帧加窗

实现 `framed_windowed(x, N, H, window)`：把信号分帧并逐帧乘上给定窗，返回 `(T, N)` 矩阵。

要求：用向量化下标（不要逐帧 python append），窗为 `None` 时用全 1（矩形窗）。

In [ ]:
def framed_windowed(x, N, H, window=None):
    # TODO:
    #  1) T = 1 + (len(x)-N)//H
    #  2) 用广播构造下标 idx[t,j] = t*H + j，取 x[idx] 得 (T,N)
    #  3) window 为 None 时用 np.ones(N)，否则乘 window
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
x = np.arange(30.0)
W = hann_window(8)
out = framed_windowed(x, N=8, H=4, window=W)
T_exp = 1 + (30-8)//4
assert out.shape == (T_exp, 8)
# 第 t 帧应等于 x[t*4 : t*4+8] * W
assert np.allclose(out[2], x[8:16]*W)
# 矩形窗(None)时就是纯切片
out_rect = framed_windowed(x, 8, 4, None)
assert np.allclose(out_rect[2], x[8:16])
print('✅ 练习 1 通过：分帧 + 加窗正确（向量化、含矩形窗分支）')

## ✏️ 练习 2：从零实现 STFT 并对拍 np.fft

复用练习 1，实现 `my_stft(x, N, H)`：分帧 + Hann 窗 + 逐帧 `rfft`，返回 `(T, N//2+1)` 复矩阵。

验证：与「手动对每个加窗帧做 `np.fft.rfft`」逐位一致。

In [ ]:
def my_stft(x, N=256, H=128):
    # TODO: 用 framed_windowed(x, N, H, hann_window(N))，再 np.fft.rfft(..., axis=1)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
x = rng.standard_normal(2000)
S = my_stft(x, 256, 128)
assert S.shape[1] == 129 and np.iscomplexobj(S)
W = hann_window(256)
for t in [0, 3, 7]:
    frame = x[t*128 : t*128+256] * W
    assert np.allclose(S[t], np.fft.rfft(frame), atol=1e-9)
print(f'✅ 练习 2 通过：my_stft 形状 {S.shape}，每帧逐位等于 np.fft.rfft(加窗帧)')

## ✏️ 练习 3：梅尔滤波器组的性质

实现 `mel_points(n_filters, sr, fmin=0, fmax=None)`：返回 `n_filters+2` 个滤波器边界点的**赫兹**坐标（在梅尔轴等距）。

验证：① 单调递增；② 端点等于 fmin/fmax；③ 体现「低频密、高频疏」（相邻间隔递增）。

In [ ]:
def mel_points(n_filters, sr, fmin=0.0, fmax=None):
    # TODO:
    #  1) fmax 默认 sr/2
    #  2) 在 [hz_to_mel(fmin), hz_to_mel(fmax)] 上 linspace 取 n_filters+2 个梅尔点
    #  3) 用 mel_to_hz 映回赫兹返回
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
pts = mel_points(8, sr=16000)
assert len(pts) == 10
assert abs(pts[0] - 0.0) < 1e-6 and abs(pts[-1] - 8000.0) < 1e-3
assert np.all(np.diff(pts) > 0), '应单调递增'
gaps = np.diff(pts)
assert gaps[-1] > gaps[0], '高频间隔应大于低频间隔（梅尔的本质）'
print('赫兹边界点:', np.round(pts, 1))
print('✅ 练习 3 通过：梅尔点低频密、高频疏')

## ✏️ 练习 4：MFCC 用 DCT 去相关

实现 `log_mel_to_mfcc(log_mel, n_mfcc)`：对 `(T, M)` 的 log-mel 沿频带做 DCT-II、取前 `n_mfcc` 个系数。

验证：① 形状 `(T, n_mfcc)`；② 对一个**沿频带线性变化**的 log-mel，能量高度集中在低阶 DCT 系数。

In [ ]:
def log_mel_to_mfcc(log_mel, n_mfcc=13):
    # TODO: dct_ii(log_mel)[:, :n_mfcc]
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
M = 40
# 一个平滑(沿频带线性)的 log-mel：DCT 后能量应集中在前几个系数
ramp = np.linspace(-2, 2, M)
log_mel = np.stack([ramp + 0.01*rng.standard_normal(M) for _ in range(5)])
coeffs = log_mel_to_mfcc(log_mel, n_mfcc=13)
assert coeffs.shape == (5, 13)
full = dct_ii(log_mel)
frac = (full[:, :4]**2).sum() / (full**2).sum()
assert frac > 0.9, '平滑信号的 DCT 能量应集中在前几个系数'
print(f'前 4 个 DCT 系数占能量 {frac:.1%}')
print('✅ 练习 4 通过：DCT 把平滑 log-mel 的能量压进低阶系数（去相关/降维）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def framed_windowed(x, N, H, window=None):
    T = 1 + max(0, (len(x) - N)//H)
    idx = np.arange(N)[None, :] + H*np.arange(T)[:, None]
    frames = x[idx]
    w = np.ones(N) if window is None else window
    return frames * w[None, :]

In [ ]:
# 练习 2 参考答案
def my_stft(x, N=256, H=128):
    frames = framed_windowed(x, N, H, hann_window(N))
    return np.fft.rfft(frames, axis=1)

In [ ]:
# 练习 3 参考答案
def mel_points(n_filters, sr, fmin=0.0, fmax=None):
    if fmax is None: fmax = sr/2
    m = np.linspace(hz_to_mel(fmin), hz_to_mel(fmax), n_filters+2)
    return mel_to_hz(m)

In [ ]:
# 练习 4 参考答案
def log_mel_to_mfcc(log_mel, n_mfcc=13):
    return dct_ii(log_mel)[:, :n_mfcc]

---
## 🧪 真实工程参数胶囊：复现 Whisper 的 log-mel 前端配置

OpenAI Whisper 的音频前端用的是**真实、公开**的一组参数。我们用这些**真实数值**算出 Whisper 输入特征的形状，并验证一段 30 秒音频会被编码成多大的 log-mel。这把你从零写的流水线直接接到了一个真实部署的模型。

（全程合成信号 + 纯 numpy，无需联网、无需 whisper 包。）

In [ ]:
# Whisper 的真实前端配置（公开于其论文与代码）
WHISPER = dict(
    sample_rate = 16000,   # 16 kHz
    n_fft       = 400,     # 25 ms 窗
    hop_length  = 160,     # 10 ms 帧移 -> 100 帧/秒
    n_mels      = 80,      # 80 维 log-mel
    chunk_sec   = 30,      # 固定 30 秒一段
    target_frames = 3000,  # Whisper 把每段 pad/截到正好 3000 帧 (=30s*100)
)

def whisper_feature_shape(cfg):
    n_samples = cfg['sample_rate'] * cfg['chunk_sec']
    # 朴素 STFT 帧数（Whisper 实现用 center padding，结果略有差异）
    n_frames = 1 + (n_samples - cfg['n_fft'])//cfg['hop_length']
    return n_frames, cfg['n_mels']

T, M = whisper_feature_shape(WHISPER)
fps = WHISPER['sample_rate'] // WHISPER['hop_length']
print(f"Whisper: 30s @ 16kHz -> 朴素帧数 {T}, 梅尔维 {M}")
print(f"帧率 = {fps} 帧/秒；Whisper 最终 pad/截到 {WHISPER['target_frames']} 帧")
assert fps == 100
assert M == 80
# 朴素帧数 2998，加 center padding 后正好 30*100=3000
assert T == 2998, '无 padding 的朴素帧数'
assert WHISPER['target_frames'] == WHISPER['chunk_sec'] * fps == 3000
print('✅ 用真实 Whisper 配置算出输入特征 (~3000 帧, 80 维 log-mel)')

**🧪 胶囊练习**：实现 `realtime_frames_per_sec(sr, hop)` 与 `feature_dims(sec, sr, hop, n_mels)`：
给定真实配置，算出 (a) 每秒多少帧；(b) `sec` 秒音频的 log-mel 总维数 (帧数 × 梅尔数)。

用它对比 Whisper(16k/160) 与一个音乐配置(44.1k/512) 的特征密度。

In [ ]:
def realtime_frames_per_sec(sr, hop):
    # TODO: 返回 sr/hop
    raise NotImplementedError

def feature_dims(sec, sr, hop, n_mels):
    # TODO: 帧数 ≈ sec*sr/hop，返回 帧数 * n_mels（取整）
    raise NotImplementedError

In [ ]:
# 自测
fps = realtime_frames_per_sec(16000, 160)
assert fps == 100, 'Whisper 配置应是 100 帧/秒'
d_speech = feature_dims(10, 16000, 160, 80)
assert d_speech == 1000*80
# 音乐配置：44.1kHz, hop 512, 128 mel
fps_music = realtime_frames_per_sec(44100, 512)
print(f'语音 16k/160 -> {fps:.0f} 帧/秒；音乐 44.1k/512 -> {fps_music:.1f} 帧/秒')
print(f'10 秒语音 log-mel 总维数 = {d_speech}')
print('✅ 胶囊练习通过：会用真实配置算特征密度')

In [ ]:
# 📖 胶囊参考答案
def realtime_frames_per_sec(sr, hop):
    return sr / hop

def feature_dims(sec, sr, hop, n_mels):
    n_frames = int(sec * sr / hop)
    return n_frames * n_mels

### 小结
- 声音 = 一维波形 + 采样率；**奈奎斯特** = sr/2，超过即不可逆混叠。
- **分帧加窗**：语音短时平稳，按 N/H 切重叠帧、乘 Hann 窗抑制泄漏；帧长定时频分辨率。
- **STFT** = 逐帧 rfft，得 (时间×频率) 复谱；取 |·| 丢相位 = 谱图。
- **梅尔滤波器组**把线性频率压成听觉频带（低密高疏）；**log-mel** 是现代语音模型主输入。
- **MFCC** = log-mel + DCT 去相关、取前 13；更紧凑，是经典强基线。

下一站：**模块 02 · ASR 与 Whisper** —— 把这些逐帧特征，对齐成文字。